In [3]:
import pandas as pd 
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from tensorflow import keras
from tensorflow.keras import layers

In [4]:
df = pd.read_csv('kalshi_initial.csv')
df['open_time'] = pd.to_datetime(df['open_time'])
df['curr_time'] = pd.to_datetime(df['curr_time'])
df['close_time'] = pd.to_datetime(df['close_time'])
df['time_to_close'] = (df['close_time'] - df['curr_time']).dt.total_seconds()
df = df.sort_values(by = ['coin', 'open_time', 'curr_time'])
df = df.drop(columns=['id', 'strike_type'])
# df = df.set_index('curr_time')
df = df.set_index('curr_time')


In [36]:
def build_coin_df(coin): 
    global df 
    df_copy = df[df['coin'] == coin].copy()
    col_identifiers = ['open_time', 'close_time']

    df_copy['next_price_dollars_lead1'] = df_copy.groupby(col_identifiers)['last_price_dollars'].shift(-1)
    df_copy = df_copy[df_copy['time_to_close'] > -0.01]

    df_copy = df_copy.add_prefix(f'{coin}_')
    
    return df_copy

In [37]:
lst = []
for i in ['BTC', 'ETH', 'XRP', 'SOL']: 
    lst.append(build_coin_df(i))

combined = pd.concat(lst, join='inner', axis=1)
# combined = combined.dropna(axis='columns')
combined.dropna(subset=['BTC_next_price_dollars_lead1'] ,inplace=True)
combined = combined.drop(columns=['BTC_coin', 'ETH_coin', 'XRP_coin', 'SOL_coin'])
time_to_close_col = combined['BTC_time_to_close']
combined['outcome'] = combined['BTC_open_time'].map((combined.sort_values(by='BTC_open_time').groupby('BTC_open_time').last()['BTC_last_price_dollars']> 0.5).astype(int))

combined = combined.select_dtypes(exclude='datetime64[us, UTC]')
combined.info() 


<class 'pandas.DataFrame'>
DatetimeIndex: 2384 entries, 2026-03-13 05:42:39.686553+00:00 to 2026-03-13 08:36:40.880525+00:00
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   BTC_last_price_dollars        2384 non-null   float64
 1   BTC_liquidity_dollars         2384 non-null   float64
 2   BTC_no_ask_dollars            2384 non-null   float64
 3   BTC_no_bid_dollars            2384 non-null   float64
 4   BTC_yes_ask_dollars           2384 non-null   float64
 5   BTC_yes_bid_dollars           2384 non-null   float64
 6   BTC_open_interest             0 non-null      float64
 7   BTC_floor_strike              2384 non-null   float64
 8   BTC_cap_strike                0 non-null      float64
 9   BTC_volume                    0 non-null      float64
 10  BTC_volume_24h_fp             2384 non-null   float64
 11  BTC_time_to_close             2384 non-null   float64
 12  BTC_next_pr

In [48]:
combined['outcome'].value_counts(normalize=True)

outcome
0    0.560403
1    0.439597
Name: proportion, dtype: float64

In [34]:
y = combined['BTC_next_price_dollars_lead1'].values
X = combined.drop(columns=['BTC_next_price_dollars_lead1']).dropna(axis='columns')
X = pd.concat([X,time_to_close_col], axis=1).values

In [35]:
n = int(X.shape[0]*0.8)

X_train, X_test = X[:n], X[n:]
y_train, y_test = y[:n], y[n:]

In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [37]:
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation='sigmoid')  # regression output
])

/Users/erxcshi/courses/ml1grad/crypto_predictions/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [39]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
)

Epoch 1/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 538us/step - loss: 0.0184 - mae: 0.0865 
Epoch 2/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 483us/step - loss: 0.0021 - mae: 0.0308
Epoch 3/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 458us/step - loss: 0.0016 - mae: 0.0252
Epoch 4/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 457us/step - loss: 0.0014 - mae: 0.0220
Epoch 5/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - loss: 0.0012 - mae: 0.0198
Epoch 6/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 450us/step - loss: 0.0011 - mae: 0.0179  
Epoch 7/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 454us/step - loss: 0.0010 - mae: 0.0171
Epoch 8/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 441us/step - loss: 9.4126e-04 - mae: 0.0160
Epoch 9/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 451us/step - loss: 9.7895e-04 - mae: 0.0162
Epoch 10/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 537us/step - loss: 9.6996e-04 - mae: 0.0162
Epoch 11/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - loss: 8.7143e-04 - mae: 0.0150
Epoch 12/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - loss: 8.7554e-04 - ma

In [40]:
pred = model.predict(X_test)
pred


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[0.01103353],
       [0.01097274],
       [0.0109125 ],
       [0.01085265],
       [0.01079284],
       [0.01073346],
       [0.01067429],
       [0.0106154 ],
       [0.01055684],
       [0.01049862],
       [0.01044071],
       [0.01038312],
       [0.01032586],
       [0.01040391],
       [0.01059135],
       [0.01053322],
       [0.01047514],
       [0.01041738],
       [0.01035993],
       [0.0103028 ],
       [0.01024599],
       [0.01018948],
       [0.01013328],
       [0.01007734],
       [0.01002176],
       [0.00996648],
       [0.00991151],
       [0.00985682],
       [0.0086402 ],
       [0.72628844],
       [0.72635573],
       [0.7264231 ],
       [0.7264904 ],
       [0.72655773],
       [0.7266251 ],
       [0.72669244],
       [0.72675955],
       [0.7268268 ],
       [0.72692156],
       [0.72760355],
       [0.7259197 ],
       [0.7402108 ],
       [0.74128944],
       [0.81442976],
       [0.72226954],
       [0.7224418 ],
       [0.7226138 ],
       [0.722

In [41]:
from sklearn.metrics import mean_squared_error as mse, r2_score as r2

print(mse(y_test, pred))
print(r2(y_test, pred))

0.0037046554724711445
0.9308502504751881
